In [2]:
import pandas as pd
import numpy as np
!pip install cassandra-driver

In [3]:
!python -c "import cassandra; print(cassandra.__version__)"

3.29.3


In [4]:
from cassandra.cluster import Cluster
from cassandra.auth import PlainTextAuthProvider

cloud_config= {
        'secure_connect_bundle':'secure-connect-proj.zip' # replace <</PATH/TO/>> with the path where your downloaded bundle was uploaded here
        }
auth_provider = PlainTextAuthProvider('mpOZPAuzqPMGuZygzsDLlLUC', 'KxOuZOmk9.+9ckeFhxoADdyCgEjLLcGrpSlYGQnQ9jYXzQejOwYvwmIx6GG_NbmG6N0G+E53gTpMq0CzcZZJOObSnmg9nN1RwpvCdvXZ611oeZel0jI1QZrIdNP5NhZO')
cluster = Cluster(cloud=cloud_config, auth_provider=auth_provider)
session = cluster.connect()


In [5]:
df = pd.read_csv("kick_starter.csv", encoding='latin-1')

In [6]:
print(df.columns)
print(df.shape)

Index(['ID', 'name', 'category', 'main_category', 'currency', 'deadline',
       'goal', 'launched', 'pledged', 'state', 'backers', 'country',
       'usd pledged', 'usd_pledged_real', 'usd_goal_real'],
      dtype='object')
(20460, 15)


In [9]:
df['launched'] = pd.to_datetime(df['launched'], format='mixed')
df['deadline'] = pd.to_datetime(df['deadline'], format='mixed')

In [ ]:
df = df[df["state"] != "live"].copy()

In [ ]:
df['CampaignDuration'] = (df['deadline'] - df['launched']).dt.days
df['FundingRatio'] = df['usd_pledged_real'] / df['usd_goal_real']
def goal_category(goal):
    if goal < 10000:
        return "Low Goal"
    elif goal <= 100000:
        return "Medium Goal"
    else:
        return "High Goal"

df['GoalCategory'] = df['usd_goal_real'].apply(goal_category)
df['LaunchMonth'] = df['launched'].dt.month


In [11]:
print(df[["CampaignDuration","FundingRatio","GoalCategory","LaunchMonth"]].head())

   CampaignDuration  FundingRatio GoalCategory  LaunchMonth
0              58.0      0.000000     Low Goal          8.0
1              59.0      0.080700  Medium Goal          9.0
2              44.0      0.004889  Medium Goal          1.0
3              29.0      0.000200     Low Goal          3.0
4              55.0      0.065795  Medium Goal          7.0


In [12]:
# Keep only 5000 rows for Cassandra
cass_df = df.head(5000).copy()

# Fill missing text values
cass_df["name"] = cass_df["name"].fillna("Unknown")
cass_df["category"] = cass_df["category"].fillna("Unknown")
cass_df["main_category"] = cass_df["main_category"].fillna("Unknown")
cass_df["currency"] = cass_df["currency"].fillna("Unknown")
cass_df["state"] = cass_df["state"].fillna("Unknown")
cass_df["country"] = cass_df["country"].fillna("Unknown")
cass_df["GoalCategory"] = cass_df["GoalCategory"].fillna("Unknown")

# Convert numeric columns safely
cass_df["ID"] = pd.to_numeric(cass_df["ID"], errors="coerce")
cass_df["goal"] = pd.to_numeric(cass_df["goal"], errors="coerce")
cass_df["pledged"] = pd.to_numeric(cass_df["pledged"], errors="coerce")
cass_df["backers"] = pd.to_numeric(cass_df["backers"], errors="coerce")
cass_df["usd_pledged_real"] = pd.to_numeric(cass_df["usd_pledged_real"], errors="coerce")
cass_df["usd_goal_real"] = pd.to_numeric(cass_df["usd_goal_real"], errors="coerce")
cass_df["CampaignDuration"] = pd.to_numeric(cass_df["CampaignDuration"], errors="coerce")
cass_df["FundingRatio"] = pd.to_numeric(cass_df["FundingRatio"], errors="coerce")
cass_df["LaunchMonth"] = pd.to_numeric(cass_df["LaunchMonth"], errors="coerce")

# Remove rows with missing required numeric values
cass_df = cass_df.dropna(subset=[
    "ID", "goal", "pledged", "backers",
    "usd_pledged_real", "usd_goal_real",
    "CampaignDuration", "FundingRatio", "LaunchMonth"
])

# Convert to final types
cass_df["ID"] = cass_df["ID"].astype(int)
cass_df["backers"] = cass_df["backers"].astype(int)
cass_df["CampaignDuration"] = cass_df["CampaignDuration"].astype(int)
cass_df["LaunchMonth"] = cass_df["LaunchMonth"].astype(int)

float_cols = ["goal", "pledged", "usd_pledged_real", "usd_goal_real", "FundingRatio"]
for col in float_cols:
    cass_df[col] = cass_df[col].astype(float)

# Convert dates to string for easier Cassandra insertion
cass_df["deadline"] = cass_df["deadline"].astype(str)
cass_df["launched"] = cass_df["launched"].astype(str)

print(cass_df.shape)
print(cass_df.dtypes)

(5000, 19)
ID                    int64
name                 object
category             object
main_category        object
currency             object
deadline             object
goal                float64
launched             object
pledged             float64
state                object
backers               int64
country              object
usd pledged         float64
usd_pledged_real    float64
usd_goal_real       float64
CampaignDuration      int64
FundingRatio        float64
GoalCategory         object
LaunchMonth           int64
dtype: object


In [13]:
print(cluster.metadata.keyspaces.keys())

dict_keys(['proj', 'system_auth', 'system_schema', 'data_endpoint_auth', 'system', 'datastax_sla', 'system_traces', 'system_views', 'system_virtual_schema'])


In [14]:
session.set_keyspace("proj")
print("Keyspace selected successfully")

Keyspace selected successfully


In [15]:
session.execute("""
CREATE TABLE IF NOT EXISTS projects (
    id int PRIMARY KEY,
    name text,
    category text,
    main_category text,
    currency text,
    deadline timestamp,
    goal double,
    launched timestamp,
    pledged double,
    state text,
    backers int,
    country text,
    usd_pledged_real double,
    usd_goal_real double,
    campaign_duration int,
    funding_ratio double,
    goal_category text,
    launch_month int
)
""")

print("Table created successfully")

Table created successfully


In [16]:
insert_query = session.prepare("""
INSERT INTO projects (
    id, name, category, main_category, currency, deadline, goal, launched,
    pledged, state, backers, country, usd_pledged_real, usd_goal_real,
    campaign_duration, funding_ratio, goal_category, launch_month
) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
""")

print("Insert statement prepared successfully")

Insert statement prepared successfully


In [17]:
from cassandra.query import BatchStatement

batch = BatchStatement()
count = 0

for _, row in cass_df.iterrows():
    batch.add(insert_query, (
        int(row["ID"]),
        str(row["name"]),
        str(row["category"]),
        str(row["main_category"]),
        str(row["currency"]),
        str(row["deadline"]),
        float(row["goal"]),
        str(row["launched"]),
        float(row["pledged"]),
        str(row["state"]),
        int(row["backers"]),
        str(row["country"]),
        float(row["usd_pledged_real"]),
        float(row["usd_goal_real"]),
        int(row["CampaignDuration"]),
        float(row["FundingRatio"]),
        str(row["GoalCategory"]),
        int(row["LaunchMonth"])
    ))

    count += 1

    if count % 100 == 0:
        session.execute(batch)
        batch = BatchStatement()

if len(batch) > 0:
    session.execute(batch)

print("Insertion completed successfully")

Insertion completed successfully


In [18]:
rows = session.execute("SELECT * FROM projects LIMIT 5")

for row in rows:
    print(row)

Row(id=1001524172, backers=20, campaign_duration=44, category='Photography', country='US', currency='USD', deadline='2011-07-24', funding_ratio=1.0036666666666667, goal=6000.0, goal_category='Low Goal', launch_month=6, launched='2011-06-09 14:43:02', main_category='Photography', name='Scar Tissue', pledged=6022.0, state='successful', usd_goal_real=6000.0, usd_pledged_real=6022.0)
Row(id=1022576045, backers=36863, campaign_duration=39, category='Product Design', country='US', currency='USD', deadline='2016-11-20', funding_ratio=102.21207, goal=10000.0, goal_category='Medium Goal', launch_month=10, launched='2016-10-11 17:57:34', main_category='Design', name='Polygons | The Flat 4-in-1 Measuring Spoon', pledged=1022120.7, state='successful', usd_goal_real=10000.0, usd_pledged_real=1022120.7)
Row(id=1002139381, backers=51, campaign_duration=39, category="Children's Books", country='US', currency='USD', deadline='2014-09-09', funding_ratio=1.261, goal=2000.0, goal_category='Low Goal', laun

In [19]:
rows = session.execute("SELECT * FROM projects")
cass_data = list(rows)

cass_df_from_db = pd.DataFrame(cass_data)

print(cass_df_from_db.shape)
cass_df_from_db.head()

(5000, 18)


,id,backers,campaign_duration,category,country,currency,deadline,funding_ratio,goal,goal_category,launch_month,launched,main_category,name,pledged,state,usd_goal_real,usd_pledged_real
0,1001524172,20,44,Photography,US,USD,2011-07-24,1.003667,6000.0,Low Goal,6,2011-06-09 14:43:02,Photography,Scar Tissue,6022.00,successful,6000.0,6022.00
1,1022576045,36863,39,Product Design,US,USD,2016-11-20,102.212070,10000.0,Medium Goal,10,2016-10-11 17:57:34,Design,Polygons | The Flat 4-in-1 Measuring Spoon,1022120.70,successful,10000.0,1022120.70
2,1002139381,51,39,Children's Books,US,USD,2014-09-09,1.261000,2000.0,Low Goal,7,2014-07-31 00:54:00,Publishing,Tales from Sheepland: from the imagination of ...,2522.00,successful,2000.0,2522.00
3,1011830662,23,21,Shorts,US,USD,2012-12-19,1.016667,1500.0,Low Goal,11,2012-11-27 05:32:37,Film & Video,Red Sky Morning,1525.00,successful,1500.0,1525.00
4,1012448004,158,29,Dance,US,USD,2012-09-16,1.009953,9000.0,Low Goal,8,2012-08-17 15:18:52,Dance,To Begin The World Over Again,9089.58,successful,9000.0,9089.58


In [20]:
q1 = cass_df_from_db.groupby("main_category").apply(
    lambda x: (x["state"].eq("successful").sum() / len(x)) * 100
).reset_index(name="success_rate_percent")

q1 = q1.sort_values(by="success_rate_percent", ascending=False)

print(q1)

   main_category  success_rate_percent
3          Dance             70.370370
14       Theater             56.849315
1         Comics             54.362416
10         Music             45.885635
0            Art             41.752577
8          Games             37.759336
6   Film & Video             36.722488
4         Design             34.020619
7           Food             29.260450
11   Photography             29.032258
12    Publishing             28.094303
5        Fashion             25.724638
2         Crafts             22.689076
9     Journalism             22.413793
13    Technology             21.359223


/tmp/ipykernel_241/3689900682.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  q1 = cass_df_from_db.groupby("main_category").apply(


In [21]:
q2 = cass_df_from_db.groupby("country")["usd_pledged_real"].mean().reset_index()

q2 = q2.sort_values(by="usd_pledged_real", ascending=False)

print(q2)

   country  usd_pledged_real
16      NL     129133.474250
10      HK      29643.352727
5       DE      22795.650741
4       CH      11707.785000
18      NZ      10631.409286
21      US      10363.471331
11      IE       9595.215333
13      LU       8612.550000
7       ES       7176.143667
9       GB       6076.853926
8       FR       5957.747778
1       AU       5361.868478
20      SG       4461.857500
3       CA       4151.383991
2       BE       3288.149091
17      NO       3098.780000
15    N,0"       2413.134151
19      SE       2407.946364
6       DK       2036.635000
12      IT       1698.643913
14      MX        547.600323
0       AT        133.208889


In [22]:
q3 = cass_df_from_db[["id", "name", "funding_ratio"]].sort_values(
    by="funding_ratio", ascending=False
).head(5)

print(q3)

              id                                               name  \
1247  1024582936  The $1 Play What You Want Mystery Campaign RPG...   
2119  1010207468  Kenophania: How A Song Is Created (Interactive...   
1871  1001685760        Wholesters- A new way to Whold your things!   
29    1002131859                               Midwest Side Stories   
4078  1009661017  Redefining Italian Luxury Watches - Filippo Lo...   

      funding_ratio  
1247    4773.910000  
2119     700.323308  
1871     566.000000  
29       253.750000  
4078     240.477358  


In [23]:
failed_df = cass_df_from_db[cass_df_from_db["state"] == "failed"]

q4 = failed_df.groupby("category").size().reset_index(name="failure_count")

q4 = q4.sort_values(by="failure_count", ascending=False)

print(q4.head(10))

           category  failure_count
110  Product Design            140
37      Documentary            108
56             Food             94
53     Film & Video             91
52          Fiction             90
141     Video Games             89
87            Music             80
50          Fashion             71
92       Nonfiction             68
124          Shorts             59


In [24]:
q5 = cass_df_from_db[cass_df_from_db["state"].isin(["successful", "failed"])]

q5 = q5.groupby("state")["campaign_duration"].mean().reset_index()

print(q5)

        state  campaign_duration
0      failed          34.307369
1  successful          30.932478


In [25]:
successful_df = cass_df_from_db[cass_df_from_db["state"] == "successful"]

q6 = successful_df.groupby("launch_month").size().reset_index(name="successful_count")

q6 = q6.sort_values(by="successful_count", ascending=False)

print(q6)

    launch_month  successful_count
4              5               176
9             10               164
10            11               163
8              9               163
6              7               161
5              6               158
7              8               158
1              2               153
3              4               152
2              3               151
0              1               107
11            12                86


In [26]:
q7 = cass_df_from_db.groupby("main_category")["backers"].mean().reset_index()

q7 = q7.sort_values(by="backers", ascending=False)

print(q7)

   main_category     backers
4         Design  367.796392
13    Technology  242.092233
8          Games  239.022822
1         Comics  119.543624
7           Food   64.250804
5        Fashion   62.202899
3          Dance   54.351852
6   Film & Video   53.718900
11   Photography   45.819355
10         Music   44.281729
12    Publishing   42.147348
9     Journalism   41.862069
0            Art   39.237113
14       Theater   37.171233
2         Crafts   21.521008


In [27]:
total_projects = len(cass_df_from_db)

successful_projects = cass_df_from_db[cass_df_from_db["funding_ratio"] >= 1].shape[0]

percentage_success = (successful_projects / total_projects) * 100

print("Percentage of projects that reached or exceeded their goal:", percentage_success)

Percentage of projects that reached or exceeded their goal: 36.720000000000006


Cassandra Schema Design Justification

Cassandra follows a wide-column model optimized for high-volume distributed data and fast reads based on primary keys. The table was designed so that each Kickstarter project is represented as a row containing both the original dataset attributes and the engineered features.

A simple primary key (id) was used to uniquely identify each project and allow efficient retrieval. Cassandra performs best when queries access rows using primary keys or indexed fields, so the schema was designed to keep all relevant attributes in a single table rather than splitting them across multiple tables.

Because Cassandra is optimized for scalability and large datasets, batch inserts were used to efficiently load the Kickstarter data into the database. This schema supports analytical queries such as calculating averages, success rates, and funding ratios while maintaining Cassandra’s distributed performance advantages.